<a href="https://colab.research.google.com/github/keenengillespie-cyber/kgillesp/blob/main/Module_5_Assignment_Chatbot_or_Sentiment_Analysis_Build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
import nltk
from nltk.corpus import stopwords
import re

# 1. Setup & Preprocessing
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^​\w\s]', '', text) # Remove punctuation, ensuring to handle zero-width space if present
    tokens = text.split()
    clean_tokens = [w for w in tokens if w not in stop_words]
    return " ".join(clean_tokens)

# 2. Load Data (Replace 'your_data.csv' with your actual file)
# Example data structure: text, label
data = {
    'text': [
        'How can I pay my bill?', 'Where is my order?', 'My laptop won\'t turn on',
        'I want to talk to a person', 'Payment options', 'Track my package',
        'Technical support needed', 'Hello, I need help'
    ],
    'label': ['billing', 'order_status', 'technical', 'support', 'billing', 'order_status', 'technical', 'support']
}
df = pd.DataFrame(data)
df['clean_text'] = df['text'].apply(preprocess_text)

# 3. Model Training
X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['label'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LinearSVC()
model.fit(X_train_tfidf, y_train)

# 4. Evaluation (Required for your project)
predictions = model.predict(X_test_tfidf)
print("Confusion Matrix:\n", confusion_matrix(y_test, predictions))
print("\nClassification Report:\n", classification_report(y_test, predictions))

# 5. Simple Fallback Logic Demo
def get_response(user_input):
    clean_input = preprocess_text(user_input)
    tfidf_input = vectorizer.transform([clean_input])
    prediction = model.predict(tfidf_input)[0]

    # Simple confidence check placeholder
    if not clean_input:
        return "I'm sorry, I didn't catch that. Can you rephrase?"

    return f"Intent detected: {prediction}"

print(get_response("I have a problem with my invoice"))

In [ ]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
import numpy as np

# 1. Prepare Data (Format your CSV into a Hugging Face Dataset)
# Assuming you have a DataFrame 'df' with columns

# Re-initialize df to ensure original string labels are used
data = {
    'text': [
        'How can I pay my bill?', 'Where is my order?', 'My laptop won\'t turn on',
        'I want to talk to a person', 'Payment options', 'Track my package',
        'Technical support needed', 'Hello, I need help'
    ],
    'label': ['billing', 'order_status', 'technical', 'support', 'billing', 'order_status', 'technical', 'support']
}
df = pd.DataFrame(data)

# Create label_map and id_map using the original string labels
original_unique_labels = sorted(df['label'].unique())

label_map = {label: i for i, label in enumerate(original_unique_labels)}
id_map = {i: label for i, label in enumerate(original_unique_labels)}

# Map DataFrame labels to integers
df['label'] = df['label'].map(label_map)

# The 'label' column in the Dataset expects standard Python integers, not numpy.int64.
# Convert df['label'] to Python int type explicitly to avoid potential issues down the line.
df['label'] = df['label'].astype(int)

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 2. Tokenization
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 3. Load Pre-trained Model
# Ensure id2label and label2id contain only Python native types (int, str)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_map),
    id2label={int(k): str(v) for k, v in id_map.items()},
    label2id={str(k): int(v) for k, v in label_map.items()}
)

# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

# Train the model
trainer.train()

# 6. Inference Pipeline (For your Demo/UI)
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

# Test it
result = classifier("My package hasn't arrived yet")
print(f"Predicted Intent: {result[0]['label']} with score {result[0]['score']:.4f}")

In [ ]:
# Install streamlit if it's not already installed
try:
    import streamlit as st
except ImportError:
    print("Streamlit not found. Installing now...")
    !pip install streamlit
    import streamlit as st # Try importing again after installation

from transformers import pipeline

# 1. Page Config
st.set_page_config(page_title="Customer Service Assistant", page_icon="🤖")
st.title("🤖 Customer Support Chatbot")
st.markdown("Submit your inquiry below to be routed to the correct department.")

# 2. Load your Model (Cached to prevent reloading on every click)
@st.cache_resource
def load_model():
    # Replace 'path/to/your/model' with your saved model folder or 'distilbert-base-uncased'
    # Make sure to load the trained model, not just the base model.
    # For now, we'll use the pre-trained 'distilbert-base-uncased' as a placeholder
    # until a properly saved and loaded trained model is available.
    return pipeline("text-classification", model="distilbert-base-uncased")

classifier = load_model()

# 3. Chat Interface
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User Input
if prompt := st.chat_input("How can I help you today?"):
    # Add user message to history
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Get model prediction
    prediction = classifier(prompt)[0]
    intent = prediction['label']
    confidence = prediction['score']

    # 4. Fallback Logic (Minimum Feature Requirement)
    if confidence < 0.6:
        response = "I'm not quite sure I understand. Could you provide more details so I can help?"
    else:
        response = f"I've categorized your request under **{intent.upper()}**. One of our specialists will be with you shortly."

    # Add bot response to history
    with st.chat_message("assistant"):
        st.markdown(response)
        st.caption(f"System Confidence: {confidence:.2%}")
    st.session_state.messages.append({"role": "assistant", "content": response})


In [ ]:
%%writefile app.py
# Install streamlit if it's not already installed
try:
    import streamlit as st
except ImportError:
    print("Streamlit not found. Installing now...")
    !pip install streamlit
    import streamlit as st # Try importing again after installation

from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

# 1. Page Config
st.set_page_config(page_title="Customer Service Assistant", page_icon="🤖")
st.title("🤖 Customer Support Chatbot")
st.markdown("Submit your inquiry below to be routed to the correct department.")

# 2. Load your Model (Cached to prevent reloading on every click)
@st.cache_resource
def load_model():
    # Load the tokenizer and model from the checkpoint directory
    model_path = "/content/results/checkpoint-3"
    # The pipeline function should automatically load the tokenizer and model configuration
    # including id2label and label2id from the saved checkpoint directory.
    return pipeline("text-classification", model=model_path)

classifier = load_model()

# 3. Chat Interface
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User Input
if prompt := st.chat_input("How can I help you today?"):
    # Add user message to history
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Get model prediction
    prediction = classifier(prompt)[0]
    intent = prediction['label']
    confidence = prediction['score']

    # 4. Fallback Logic (Minimum Feature Requirement)
    if confidence < 0.6:
        response = "I'm not quite sure I understand. Could you provide more details so I can help?"
    else:
        response = f"I've categorized your request under **{intent.upper()}**. One of our specialists will be with you shortly."

    # Add bot response to history
    with st.chat_message("assistant"):
        st.markdown(response)
        st.caption(f"System Confidence: {confidence:.2%}")
    st.session_state.messages.append({"role": "assistant", "content": response})


In [ ]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.228.121.193:8501

